# NIDS CNN-LSTM Autoencoder (Colab Ready)

This notebook runs the full pipeline: preprocess → train → evaluate (CIC + CSE).

**Folder layout expected in Google Drive:**
```
MyDrive/nids-cnn-lstm-autoencoder/
  data/raw/CIC-IDS2017/
  data/raw/CSE-CIC-IDS2018/
```

If your dataset is elsewhere, update the paths in `config.yaml`.


Dependency install step in this notebook prefers `uv` (with automatic fallback to `pip`).


> Note: Untuk lokal Windows (DirectML), default konfigurasi project ini memakai CPU (`training.force_cpu: true`) demi stabilitas. Jika ingin akselerasi GPU, gunakan runtime Google Colab GPU.


In [2]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('[SKIP] Bukan runtime Colab, mount Drive dilewati.')


[SKIP] Bukan runtime Colab, mount Drive dilewati.


In [2]:
# Move to project root (robust local/colab)
import os
from pathlib import Path

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _is_project_root(p: Path) -> bool:
    return (p / 'requirements.txt').exists() and (p / 'scripts').exists()

def _find_project_root() -> Path:
    env_root = os.environ.get('NIDS_PROJECT_ROOT', '').strip()
    if env_root:
        p = Path(env_root)
        if _is_project_root(p):
            return p

    # quick local candidates
    candidates = [Path.cwd(), *Path.cwd().parents]

    # colab candidates
    mydrive = Path('/content/drive/MyDrive')
    expected = mydrive / 'nids-cnn-lstm-autoencoder'
    expected_colab_nb = mydrive / 'Colab Notebooks' / 'nids-cnn-lstm-autoencoder'
    candidates.extend([expected_colab_nb, expected])

    for c in candidates:
        if _is_project_root(c):
            return c

    # deep scan only in Colab MyDrive
    if _is_colab_runtime() and mydrive.exists():
        for req in mydrive.rglob('requirements.txt'):
            c = req.parent
            if _is_project_root(c):
                return c

    raise FileNotFoundError(
        'Project root not found. Set env NIDS_PROJECT_ROOT or place repo under /content/drive/MyDrive.'
    )

PROJECT_ROOT = _find_project_root().resolve()
os.chdir(PROJECT_ROOT)

# ensure working dirs exist
for rel in [
    'data/raw/CIC-IDS2017',
    'data/raw/CSE-CIC-IDS2018',
    'data/processed',
    'models',
    'results',
]:
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CWD:', Path.cwd())
print('CIC CSV count:', len(list((PROJECT_ROOT / 'data/raw/CIC-IDS2017').glob('*.csv'))))
print('CSE CSV count:', len(list((PROJECT_ROOT / 'data/raw/CSE-CIC-IDS2018').glob('*.csv'))))


PROJECT_ROOT: D:\Codingan\python\nids-cnn-lstm-autoencoder
CWD: D:\Codingan\python\nids-cnn-lstm-autoencoder
CIC CSV count: 8
CSE CSV count: 3


In [3]:
# Install dependencies (Colab-aware: local pins vs colab runtime)
import sys
import shutil
import subprocess
from pathlib import Path

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

req = Path('requirements.txt').resolve()
if not req.exists():
    raise FileNotFoundError(f'requirements.txt not found at {req}')

def _run(cmd):
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

def _install_with_pip(req_path: Path):
    _run([sys.executable, '-m', 'pip', 'install', '-r', str(req_path)])

def _install_with_uv(req_path: Path):
    _run(['uv', 'pip', 'install', '-r', str(req_path)])

is_colab = _is_colab_runtime()

# In Colab, avoid Windows-local pins (tensorflow==2.10.1, numpy<2)
effective_req = req
if is_colab:
    filtered = []
    for line in req.read_text(encoding='utf-8').splitlines():
        s = line.strip()
        if not s or s.startswith('#'):
            filtered.append(line)
            continue
        low = s.lower()
        if low.startswith('tensorflow') or low.startswith('numpy'):
            continue
        filtered.append(line)

    effective_req = Path('/tmp/requirements_colab.txt')
    effective_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
    print('[INFO] Colab mode: using filtered requirements (skip local tensorflow/numpy pins).')

used_uv = False
if shutil.which('uv'):
    try:
        _install_with_uv(effective_req)
        used_uv = True
    except subprocess.CalledProcessError:
        print('[WARN] uv install failed, falling back to pip...')

if not used_uv:
    try:
        if not shutil.which('uv'):
            _run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
            try:
                _install_with_uv(effective_req)
                used_uv = True
            except subprocess.CalledProcessError:
                print('[WARN] uv still failed after install, using pip...')
    except subprocess.CalledProcessError:
        print('[WARN] failed to install uv, using pip...')

if not used_uv:
    _install_with_pip(effective_req)

# Ensure TensorFlow exists in Colab after filtered install
if is_colab:
    try:
        import tensorflow as tf  # noqa: F401
        print('[INFO] TensorFlow already available in Colab.')
    except Exception:
        print('[INFO] TensorFlow not found, installing latest compatible build...')
        _run([sys.executable, '-m', 'pip', 'install', 'tensorflow'])


[CMD] uv pip install -r D:\Codingan\python\nids-cnn-lstm-autoencoder\requirements.txt


In [4]:
# Optional: Show GPU
import sys
import time
import shlex
import subprocess
import tensorflow as tf

print(sys.executable)
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

def _fmt_duration(seconds: float) -> str:
    sec = max(0, int(seconds))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h}h {m:02d}m {s:02d}s'
    if m > 0:
        return f'{m}m {s:02d}s'
    return f'{s}s'

def _normalize_cmd(cmd: str):
    parts = shlex.split(cmd, posix=False)
    if parts and parts[0].lower() == 'python':
        parts[0] = sys.executable
    return parts

def run_stage(stage_no: int, total_stages: int, title: str, commands):
    if isinstance(commands, str):
        commands = [commands]

    start_pct = ((stage_no - 1) / total_stages) * 100
    end_pct = (stage_no / total_stages) * 100
    print(f'[STAGE {stage_no}/{total_stages}] {title} started | global progress {start_pct:.0f}% -> {end_pct:.0f}%')

    t0 = time.time()
    for i, cmd in enumerate(commands, start=1):
        args = _normalize_cmd(cmd)
        print('[PROGRESS] command {}/{}: {}'.format(i, len(commands), ' '.join(args)))
        subprocess.run(args, check=True, cwd=str(PROJECT_ROOT))

    print(f'[DONE] Stage {stage_no}/{total_stages} ({title}) finished in {_fmt_duration(time.time() - t0)}')


d:\Codingan\python\nids-cnn-lstm-autoencoder\.venv\Scripts\python.exe
2.10.1
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## Optional (Colab only): Download Dataset from Kaggle
Cell ini **opsional** dan default-nya OFF. Jalankan hanya saat runtime Google Colab.

Syarat: simpan token Kaggle di `MyDrive/kaggle.json`.


In [ ]:
# Optional Kaggle downloader (Colab only)
from pathlib import Path
import sys
import os
import shutil
import subprocess
import zipfile

ENABLE_KAGGLE_DOWNLOAD = True  # ubah ke True kalau ingin download
OVERWRITE_EXISTING = True      # True kalau ingin timpa file lama

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _run(cmd):
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if not _is_colab_runtime():
    print('[SKIP] Bukan runtime Colab. Cell download Kaggle tidak dijalankan.')
elif not ENABLE_KAGGLE_DOWNLOAD:
    print('[SKIP] ENABLE_KAGGLE_DOWNLOAD=False. Aktifkan jika ingin download dataset.')
else:
    project_root = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path(os.getcwd())
    raw_cic = project_root / 'data' / 'raw' / 'CIC-IDS2017'
    raw_cse = project_root / 'data' / 'raw' / 'CSE-CIC-IDS2018'
    raw_cic.mkdir(parents=True, exist_ok=True)
    raw_cse.mkdir(parents=True, exist_ok=True)

    # Setup kaggle token
    drive_token = Path('/content/drive/MyDrive/Colab Notebooks/kaggle.json')
    if not drive_token.exists():
        raise FileNotFoundError('kaggle.json tidak ditemukan di /content/drive/MyDrive/Colab Notebooks/kaggle.json')

    _run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_target = kaggle_dir / 'kaggle.json'
    shutil.copy2(drive_token, token_target)
    os.chmod(token_target, 0o600)

    tmp = Path('/content/kaggle_tmp')
    tmp.mkdir(parents=True, exist_ok=True)

    def download_and_extract(dataset_slug: str, zip_name: str, out_dir: Path):
        zip_path = tmp / zip_name
        _run(['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', str(tmp), '--force'])
        if not zip_path.exists():
            # fallback: kaggle kadang pakai nama file berdasarkan slug
            candidates = sorted(tmp.glob('*.zip'), key=lambda x: x.stat().st_mtime, reverse=True)
            if not candidates:
                raise FileNotFoundError(f'Zip tidak ditemukan untuk {dataset_slug}')
            zip_path = candidates[0]
        extract_dir = tmp / (zip_path.stem + '_extract')
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_dir)

        csvs = sorted(extract_dir.rglob('*.csv'))
        copied = 0
        for f in csvs:
            dst = out_dir / f.name
            if dst.exists() and not OVERWRITE_EXISTING:
                continue
            shutil.copy2(f, dst)
            copied += 1
        print(f'[DONE] {dataset_slug}: copied {copied} CSV files to {out_dir}')

    # CIC-IDS2017 mirror (Kaggle)
    download_and_extract('chethuhn/network-intrusion-dataset', 'network-intrusion-dataset.zip', raw_cic)

    # CSE-CIC-IDS2018 mirror (Kaggle)
    download_and_extract('solarmainframe/ids-intrusion-csv', 'ids-intrusion-csv.zip', raw_cse)

    print('[SUMMARY] CIC CSV:', len(list(raw_cic.glob('*.csv'))))
    print('[SUMMARY] CSE CSV:', len(list(raw_cse.glob('*.csv'))))


[CMD] /usr/bin/python3 -m pip install -q kaggle
[CMD] kaggle datasets download -d chethuhn/network-intrusion-dataset -p /content/kaggle_tmp --force
[DONE] chethuhn/network-intrusion-dataset: copied 8 CSV files to /content/drive/MyDrive/Colab Notebooks/nids-cnn-lstm-autoencoder/data/raw/CIC-IDS2017
[CMD] kaggle datasets download -d solarmainframe/ids-intrusion-csv -p /content/kaggle_tmp --force
[DONE] solarmainframe/ids-intrusion-csv: copied 10 CSV files to /content/drive/MyDrive/Colab Notebooks/nids-cnn-lstm-autoencoder/data/raw/CSE-CIC-IDS2018
[SUMMARY] CIC CSV: 8
[SUMMARY] CSE CSV: 10


## 1) Preprocess
Generates (default sharded mode):
- data/processed/shards/cic/train/*.npz + manifest.json
- data/processed/shards/cic/val/*.npz + manifest.json
- data/processed/shards/cic/test/*.npz + manifest.json
- data/processed/shards/cse/test/*.npz + manifest.json
- data/processed/scaler.pkl
- data/processed/feature_columns.json


In [7]:
run_stage(1, 4, 'Preprocess', 'python scripts/preprocess.py --config config.yaml')


[STAGE 1/4] Preprocess started | global progress 0% -> 25%
[PROGRESS] command 1/1: /usr/bin/python3 scripts/preprocess.py --config config.yaml


CalledProcessError: Command '['/usr/bin/python3', 'scripts/preprocess.py', '--config', 'config.yaml']' died with <Signals.SIGKILL: 9>.

## 2) Train Hybrid CNN-LSTM Autoencoder


In [5]:
# run_stage(2, 4, 'Train Hybrid CNN-LSTM AE', 'python scripts/train_cnn_lstm_ae.py --config config.yaml')

import sys, subprocess

cmd = [sys.executable, "scripts/train_cnn_lstm_ae.py", "--config", "config.yaml"]

proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

ret = proc.wait()
print(f"\n[EXIT CODE] {ret}")
if ret != 0:
    raise RuntimeError(f"Training failed. Last log line: {line.strip()}")

2026-02-15 06:41:30.475396: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'cudart64_110.dll'; dlerror: cudart64_110.dll not found
2026-02-15 06:41:30.475643: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-02-15 06:41:32.147649: I tensorflow/c/logging.cc:34] Successfully opened dynamic library d:\Codingan\python\nids-cnn-lstm-autoencoder\.venv\lib\site-packages\tensorflow-plugins/directml/directml.d6f03b303ac3c4f2eeb8ca631688c9757b361310.dll
2026-02-15 06:41:32.148997: I tensorflow/c/logging.cc:34] Successfully opened dynamic library dxgi.dll
2026-02-15 06:41:32.173710: I tensorflow/c/logging.cc:34] Successfully opened dynamic library d3d12.dll
2026-02-15 06:41:32.598472: I tensorflow/c/logging.cc:34] DirectML device enumeration: found 2 compatible adapters.
2026-02-15 06:41:33,279 | INFO | [STAGE] force_cpu=true -> training will run on CPU.
2026-02-15

KeyboardInterrupt: 

## 3) Evaluate (CIC + CSE)
Outputs metrics + plots in `results/` using the same threshold logic from config (percentile on CIC validation BENIGN).

Example outputs for `--tag cnn_lstm`:
- results/metrics/cnn_lstm_cic_metrics.json
- results/metrics/cnn_lstm_cse_metrics.json
- results/metrics/cnn_lstm_generalization_gap.json
- results/plots/cnn_lstm/roc_cic.png
- results/plots/cnn_lstm/roc_cse.png


In [ ]:
run_stage(3, 4, 'Evaluate (CIC + CSE)', 'python scripts/eval_metrics.py --config config.yaml --model models/cnn_lstm_ae/best_model.keras --tag cnn_lstm')


## 4) Optional: Train LSTM Autoencoder Baseline


In [ ]:
run_stage(4, 4, 'Baseline LSTM AE', [
    'python scripts/train_lstm_ae.py --config config.yaml',
    'python scripts/eval_metrics.py --config config.yaml --model models/lstm_ae/best_model.keras --tag lstm_ae',
])
